<a href="https://colab.research.google.com/github/Nithya2405/ai-engineering-workbench/blob/main/NexusNote.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Running **Streamlit** inside Google Colab is a bit different than running it on your local computer because Colab is a "closed" environment.
### 🛠 Step 1: Install the Required Libraries

Create a new cell and run this to install everything needed for the frontend, the database, and the AI.

In [1]:
!pip install -q streamlit google-genai chromadb sentence-transformers groq
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
changed 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇



### 🛠 Step 2: Write the `app.py` File

In Colab, we use the `%%writefile` magic command. This creates a physical file in your Colab sidebar that Streamlit can run.

**Run this cell:**

In [4]:
%%writefile app.py
import streamlit as st
from groq import Groq # Switching to Groq for speed & quota stability
from sentence_transformers import SentenceTransformer
import chromadb
from google.colab import userdata

# --- App Config ---
st.set_page_config(page_title="NexusNote: AI Workbench", layout="wide")
st.title("📂 NexusNote: Chat with Your Docs")
st.markdown("---")

# --- 1. Sidebar Setup (API Keys) ---
with st.sidebar:
    st.header("Settings")
    api_key = st.text_input("Enter Groq API Key", type="password")
    if not api_key:
        st.warning("Please enter your API key to start.")

# --- 2. Initialize Models ---
@st.cache_resource
def load_models():
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')
    db_client = chromadb.Client()
    return embed_model, db_client

if api_key:
    client = Groq(api_key=api_key)
    embed_model, db_client = load_models()
    collection = db_client.get_or_create_collection("user_docs")

    # --- 3. File Upload & Indexing ---
    uploaded_file = st.file_uploader("Upload a text file (e.g., Syllabus.txt)", type="txt")

    if uploaded_file:
        raw_text = uploaded_file.read().decode("utf-8")
        # Chunking by double newlines (paragraphs)
        chunks = [c.strip() for c in raw_text.split("\n\n") if c.strip()]

        if st.button("Build Knowledge Base"):
            with st.spinner("Processing documents into Vector DB..."):
                collection.add(
                    documents=chunks,
                    ids=[f"id{i}" for i in range(len(chunks))],
                    embeddings=embed_model.encode(chunks).tolist()
                )
            st.success(f"Indexed {len(chunks)} sections!")

    # --- 4. Chat Interface (The RAG Logic) ---
    st.subheader("💬 Ask a Question")
    query = st.chat_input("What is in the document?")

    if query:
        st.chat_message("user").write(query)

        # RETRIEVE
        query_vec = embed_model.encode(query).tolist()
        results = collection.query(query_embeddings=[query_vec], n_results=1)
        context = results['documents'][0][0]

        # GENERATE (Groq Llama 3)
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": f"Use ONLY this context: {context}"},
                {"role": "user", "content": query}
            ],
            model="llama-3.3-70b-versatile",
        )

        with st.chat_message("assistant"):
            st.write(chat_completion.choices[0].message.content)
            with st.expander("See Retrieved Context"):
                st.info(context)

Overwriting app.py


---

### 🛠 Step 3: Launch the App & The Tunnel

Ngrok is way more stable for Streamlit. Follow these steps exactly:

#### 1. Get your Free Token

* Go to [ngrok.com](https://ngrok.com/) and sign up (it takes 30 seconds).
* Go to your **Dashboard** and copy your **Authtoken**.

This will kill the old Localtunnel and start a stable Ngrok tunnel.
**Run this cell:**

In [6]:
# 1. Install Ngrok wrapper
!pip install -q pyngrok

from pyngrok import ngrok
import os

# 2. Terminate any existing tunnels
ngrok.kill()

# 3. Authenticate (PASTE YOUR TOKEN HERE)
NGROK_AUTH_TOKEN = "380vtYzP2rYzDQG9RcrG6QMKR88_52XuS8JpcxTgX1GWRniqz"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 4. Run Streamlit in the background
os.system("streamlit run app.py &")

# 5. Open the tunnel
public_url = ngrok.connect(8501)
print(f"🚀 NexusNote is officially live at: {public_url}")

🚀 NexusNote is officially live at: NgrokTunnel: "https://aileen-lendable-unarguably.ngrok-free.dev" -> "http://localhost:8501"
